In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.signal import find_peaks

In [ ]:
profit_matrix = pd.read_csv('../data/processed/profit_matrix.csv')
profit_matrix['time'] = pd.to_datetime(profit_matrix['time'])
profit_matrix['date'] = profit_matrix['time'].dt.date
profit_matrix['time'] = profit_matrix['time'].dt.time
trades_per_days = profit_matrix.groupby('date').size().reset_index(name = 'number_of_trades')
returns_cols = profit_matrix.drop(columns = ['time', 'date']).columns

In [ ]:
fig = plt.figure(figsize = (15, 10), dpi = 800)
plt.bar(trades_per_days['date'], trades_per_days['number_of_trades'], 
        width = 0.7, color = 'green', alpha = 0.8)
plt.title('Trades per day in period 2014-2025', fontsize = 16, fontweight = 'bold', pad = 15)
plt.xlabel('Date', fontsize = 12)
plt.ylabel('Trades per day', fontsize = 12)
plt.xlim(trades_per_days['date'].min(), trades_per_days['date'].max())
plt.ylim(trades_per_days['number_of_trades'].min(), trades_per_days['number_of_trades'].max())
rolling_mean = trades_per_days['number_of_trades'].rolling(window = 30).mean()
plt.plot(trades_per_days['date'], rolling_mean, color = 'red', 
         linewidth = 2, label = 'Moving average 30 days')
plt.grid(axis = 'y', linestyle = '--', alpha = 0.4)
plt.legend()
plt.tight_layout(pad = 0.5)
fig.savefig('../reports/images/trades.png', dpi = 150)
plt.close(fig)

In [ ]:
trades_per_days['date'] = pd.to_datetime(trades_per_days['date'])
trades_per_days['day_of_week'] = trades_per_days['date'].dt.day_name()
trades_per_days['year'] = trades_per_days['date'].dt.year
weekday_per_years_data = trades_per_days.groupby(['year', 'day_of_week'])['number_of_trades'].mean().unstack()
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_per_years_data = weekday_per_years_data[order]
weekday_columns = weekday_per_years_data.drop(columns = ['Saturday', 'Sunday']).columns

In [ ]:
fig = plt.figure(figsize = (15, 10), dpi = 800)
sns.heatmap(weekday_per_years_data[weekday_columns], 
            annot = True, fmt = '.0f', cmap = 'Blues',
            cbar_kws = {'label': 'Average number of trades'})
plt.title('Average number of trades for years per days in 2014-2025 period', fontsize = 14, fontweight = 'bold')
plt.xlabel('Day of week', fontsize = 12)
plt.ylabel('Year', fontsize = 12)
plt.tight_layout()
fig.savefig('../reports/images/avg_trades_years.png', dpi = 150)
plt.close(fig)

In [ ]:
fig = plt.figure(figsize = (14, 8), dpi = 800)
daily_counts_weekends_dropped = trades_per_days[~trades_per_days['day_of_week'].isin(['Saturday', 'Sunday'])]
sns.boxplot(data = daily_counts_weekends_dropped, 
            x = 'year', y = 'number_of_trades', 
            hue = 'day_of_week', 
            hue_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday'])
plt.title('Boxplots of trade`s numbers for years per weekdays', fontsize = 14, fontweight = 'bold')
plt.legend(bbox_to_anchor = (1.05, 1), loc = 'upper left')
plt.tight_layout()
fig.savefig('../reports/images/trades_boxplots.png', dpi = 150)
plt.close(fig)

In [ ]:
daily_returns = profit_matrix.groupby('date')[returns_cols].mean().sum(axis = 1)
cumulative = (1 + daily_returns).cumprod()
running_max = cumulative.expanding().max()
drawdown = (cumulative - running_max) / running_max

peaks, _ = find_peaks(-drawdown, distance = 365)
peak_dates = drawdown.index[peaks]
peak_values = drawdown.iloc[peaks]

fig = plt.figure(figsize = (15, 10))
plt.fill_between(drawdown.index, drawdown * 100, 0, color = 'red', alpha = 0.4)
for i, (date, value) in enumerate(zip(peak_dates, peak_values)):
    x_norm = (date - drawdown.index.min()) / (drawdown.index.max() - drawdown.index.min())
    plt.axhline(y = value * 100, xmin = x_norm - 0.015, xmax = x_norm, color = 'green', linestyle = '-.', linewidth = 1.2)
    plt.text(date, value * 100, f'{value * 100:.1f}%%\n{date.strftime("%Y-%m-%d")}', ha = 'left', va = 'top', fontsize = 8, fontweight = 'bold')
    
plt.title('Drawdowns of investment portfolio', fontsize = 14, fontweight = 'bold')
plt.ylabel('Drawdown, %')
plt.ylim(bottom = drawdown.min() * 105, top = 0)
range_time = drawdown.index.max() - drawdown.index.min()
plt.xlim(drawdown.index.min() - range_time * 0.001, drawdown.index.max() + range_time * 0.001)
plt.grid(True, alpha = 0.3)
plt.tight_layout()
fig.savefig('../reports/images/drawdowns.png', dpi = 150)
plt.close(fig)